# Quick check: model, dataset, one batch

Run from the repo root. For a real backbone switch the config to `configs/train.yaml`.

In [ ]:
import sys, os
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) == 'notebooks' else None
sys.path.insert(0, os.getcwd())

import torch
from src.train import load_config
from src.model import TraceDetector
from src.losses import trace_loss
from src.data import AlignedPairsDataset
from torch.utils.data import DataLoader

cfg = load_config('configs/smoke.yaml')   # tiny random ViT on CPU; use configs/train.yaml for DINOv3

In [ ]:
# model
model = TraceDetector.from_config(cfg['model'])
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f'patch {model.patch_size}, trainable {n_train/1e6:.2f}M of {n_total/1e6:.1f}M')

In [ ]:
# dataset: aligned pairs (real + re-decoded copies), see src/data/prepare_pairs.py
# the smoke test builds a synthetic pairs folder; here the csv must exist
dataset = AlignedPairsDataset(cfg['data']['pairs_csv'], cfg['data'], split='train')
loader = DataLoader(dataset, batch_size=2, shuffle=True)
batch = next(iter(loader))
batch['image'].shape, batch['target'].shape, batch['label'], batch['mode']

In [ ]:
# forward + loss
out = model(batch['image'])
losses = trace_loss(out['patch_logits'], out['image_logit'], batch['target'], cfg['loss'])
out['patch_logits'].shape, {k: round(v.item(), 4) for k, v in losses.items()}

In [ ]:
# look at a sample and its target map
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(batch['image'][0].permute(1, 2, 0)); ax[0].set_title(batch['mode'][0])
ax[1].imshow(batch['target'][0, 0], vmin=0, vmax=1, cmap='magma'); ax[1].set_title('target')
for a in ax: a.axis('off')